# 01 — Fitting a Prism Spectrum

This notebook demonstrates the basic `jwspecfit` workflow:
1. Load a JWST NIRSpec PRISM spectrum from FITS
2. Fit all observable emission lines
3. Inspect per-line results (flux, SNR, EW)
4. Plot the fit

The prism has wavelength-dependent resolution R(λ) ≈ 30–300, which
`jwspecfit` handles automatically from the FITS header.

In [ ]:
import jwspecfit
import matplotlib.pyplot as plt

print(f"jwspecfit v{jwspecfit.__version__}")

## Load the spectrum

The FITS file has a `SPEC1D` HDU with columns `wave` (µm), `flux` (µJy),
and `err` (µJy).  The grating is read from the header automatically.

In [ ]:
spec = jwspecfit.read_fits("../../data/borg-v4_prism-clear_1747_732.spec.fits", z=6.0)

print(f"Grating:    {spec.grating}")
print(f"Pixels:     {spec.n_pix}")
print(f"Wave range: {spec.wave_um.min():.3f} – {spec.wave_um.max():.3f} µm")

## Quick look at the raw spectrum

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
valid = spec.mask_valid()
ax.step(spec.wave_um[valid], spec.flux_ujy[valid], where="mid", lw=0.8, color="0.3")
ax.fill_between(
    spec.wave_um[valid],
    (spec.flux_ujy - spec.err_ujy)[valid],
    (spec.flux_ujy + spec.err_ujy)[valid],
    step="mid", alpha=0.15, color="0.5",
)
ax.set_xlabel(r"Wavelength [$\mu$m]")
ax.set_ylabel(r"Flux density [$\mu$Jy]")
ax.set_title("Raw PRISM spectrum")
plt.tight_layout()

## Fit emission lines

A single call does everything: continuum subtraction, line detection,
Gaussian fitting with resolution-aware bounds.  The grating and
resolution are auto-detected from the FITS header.

In [ ]:
result = jwspecfit.fit_lines(spec, z=6.0)

print(f"Fit success: {result.success}")
print(f"χ²/dof:      {result.chi2:.2f}")
print(f"Lines:       {len(result.lines)}")

## Per-line results

Each line has flux, uncertainty, SNR, equivalent width, centroid, and width.

In [ ]:
print(f"{'Line':<18s} {'Flux':>12s} {'Flux err':>12s} {'SNR':>8s} {'EW (Å)':>10s} {'σ (Å)':>8s}")
print("-" * 72)
for name, lr in result.lines.items():
    print(
        f"{name:<18s} {lr.flux:12.3e} {lr.flux_err:12.3e} {lr.snr:8.1f} {lr.ew_A:10.1f} {lr.sigma_A:8.1f}"
    )

## Plot the fit

In [ ]:
fig = jwspecfit.plot_fit(result)
plt.show()

## Bootstrap uncertainties

For more robust errors, pass `n_boot=200` (takes ~30 s).

In [ ]:
result_boot = jwspecfit.fit_lines(spec, z=6.0, n_boot=100)

print(f"{'Line':<18s} {'Flux':>12s} {'Boot err':>12s} {'SNR':>8s}")
print("-" * 54)
for name, lr in result_boot.lines.items():
    if lr.snr > 2:
        print(f"{name:<18s} {lr.flux:12.3e} {lr.flux_err:12.3e} {lr.snr:8.1f}")

## Fit specific lines only

You can restrict the fit to a subset of lines:

In [ ]:
result_oiii = jwspecfit.fit_lines(
    spec, z=6.0,
    lines=["OIII_4959", "OIII_5007", "HBETA"],
)
fig = jwspecfit.plot_fit(result_oiii)
plt.show()